# 1. 从零实现 Mini-Conformer（离线）

**研究问题**：80 维声学特征经过下采样和 Conformer block 时，时间长度、通道数和局部/全局信息怎样变化？

**方法**：用可运行的教学模型逐模块追踪张量。输入是随机合成特征，不涉及真实 FBank、训练或识别率。这里采用绝对正弦位置编码和普通多头注意力；原版 Conformer 使用相对位置注意力。对应练习：`conformer_relative_attention`、`conformer_conv`、`conformer_block`。

## 整体架构图




In [3]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(7)
B, T, FREQ, D = 2, 101, 80, 32

## 1. 卷积下采样

两层 `Conv2d(kernel=3, stride=2, padding=1)` 分别压缩时间和频率。每层输出长度为 $L' = \lfloor(L+2-3)/2\rfloor+1=\lceil L/2\rceil$。因此时间维是 $T_2=\lceil\lceil T/2\rceil/2\rceil=\lceil T/4\rceil$，并非对任意长度都等于 `T//4`。

形状：`[B,T,80] → [B,1,T,80] → [B,D,T₁,40] → [B,D,T₂,20] → [B,T₂,20D] → [B,T₂,D]`。

In [4]:
class ConvSubsampling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.conv1 = nn.Conv2d(1, d_model, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(d_model, d_model, 3, stride=2, padding=1)
        self.proj = nn.Linear(d_model * 20, d_model)  # 输入频率固定为 80

    def forward(self, x, trace=False):
        shapes = [tuple(x.shape)]
        x = x.unsqueeze(1); shapes.append(tuple(x.shape))
        x = F.silu(self.conv1(x)); shapes.append(tuple(x.shape))
        x = F.silu(self.conv2(x)); shapes.append(tuple(x.shape))
        x = x.transpose(1, 2).flatten(2); shapes.append(tuple(x.shape))
        x = self.proj(x); shapes.append(tuple(x.shape))
        return (x, shapes) if trace else x

sub = ConvSubsampling(D)
x = torch.randn(B, T, FREQ)
h, shapes = sub(x, trace=True)
print(shapes)
assert h.shape == (B, math.ceil(T / 4), D)

[(2, 101, 80), (2, 1, 101, 80), (2, 32, 51, 40), (2, 32, 26, 20), (2, 26, 640), (2, 26, 32)]


## 2. 位置编码与注意力

正弦位置编码 $PE_{t,2i}=\sin(t/10000^{2i/D})$、$PE_{t,2i+1}=\cos(t/10000^{2i/D})$，逐位置加到 `[B,S,D]`，形状不变。每头维度 $d_h=D/H$。$Q,K,V:[B,H,S,d_h]$，分数 $QK^\top/\sqrt{d_h}:[B,H,S,S]$，沿 key 轴 softmax 后乘 $V$ 得 `[B,H,S,d_h]`，合并头为 `[B,S,D]`。离线注意力可看到整段 $S$ 帧；计算和权重存储随 $S^2$ 增长。

In [5]:
class SinusoidalPE(nn.Module):
    def forward(self, x):
        b, s, d = x.shape
        pos = torch.arange(s, device=x.device, dtype=x.dtype)[:, None]
        scale = torch.exp(torch.arange(0, d, 2, device=x.device, dtype=x.dtype) * (-math.log(10000.0) / d))
        pe = torch.zeros(s, d, device=x.device, dtype=x.dtype)
        pe[:, 0::2] = torch.sin(pos * scale)
        pe[:, 1::2] = torch.cos(pos * scale)
        return x + pe[None]

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, heads=4):
        super().__init__()
        assert d_model % heads == 0
        self.heads = heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, trace=False):
        b, s, d = x.shape; dh = d // self.heads
        q, k, v = self.qkv(x).reshape(b, s, 3, self.heads, dh).permute(2, 0, 3, 1, 4)
        score = q @ k.transpose(-2, -1) / math.sqrt(dh)
        weights = score.softmax(-1)
        y = (weights @ v).transpose(1, 2).reshape(b, s, d)
        y = self.out(y)
        return (y, {'qkv': tuple(q.shape), 'scores': tuple(score.shape), 'output': tuple(y.shape)}) if trace else y

p = SinusoidalPE()(h)
a, att_shapes = MultiHeadSelfAttention(D)(p, trace=True)
print(att_shapes)
assert p.shape == a.shape == h.shape

{'qkv': (2, 4, 26, 8), 'scores': (2, 4, 26, 26), 'output': (2, 26, 32)}


## 3. Macaron FFN 与卷积模块

FFN 在每个时间位置独立计算 $W_2\,\mathrm{SiLU}(W_1x)$：`[B,S,D] → [B,S,4D] → [B,S,D]`。Conformer 在注意力前后各放一个 FFN，分别乘 $1/2$ 后作残差。

卷积模块先做 `LayerNorm → 1×1 Conv`，产生 `[B,2D,S]`；按通道拆为 $A,G:[B,D,S]$，GLU 为 $A\odot\sigma(G)$。深度卷积逐通道沿时间计算 $u_{b,c,t}=\sum_{r=0}^{K-1} w_{c,r}g_{b,c,t+r-p}$，其中 $p=(K-1)/2$；这里使用零填充，输出仍为 `[B,D,S]`。随后 `BatchNorm → SiLU → 1×1 Conv` 恢复 `[B,S,D]`。BatchNorm 在训练模式依赖当前 batch 的统计量；第二本的流式等价实验会使用独立的、因果的教学模块。

In [6]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 4*d_model), nn.SiLU(), nn.Linear(4*d_model, d_model))
    def forward(self, x): return self.net(x)

class ConvModule(nn.Module):
    def __init__(self, d_model, kernel=7):
        super().__init__()
        assert kernel % 2 == 1
        self.norm = nn.LayerNorm(d_model)
        self.pw1 = nn.Conv1d(d_model, 2*d_model, 1)
        self.dw = nn.Conv1d(d_model, d_model, kernel, padding=kernel//2, groups=d_model)
        self.bn = nn.BatchNorm1d(d_model)
        self.pw2 = nn.Conv1d(d_model, d_model, 1)
    def forward(self, x, trace=False):
        z = self.norm(x).transpose(1, 2)
        z = self.pw1(z); pw_shape = tuple(z.shape)
        z = F.glu(z, dim=1); glu_shape = tuple(z.shape)
        z = self.dw(z); dw_shape = tuple(z.shape)
        z = self.pw2(F.silu(self.bn(z))).transpose(1, 2)
        return (z, {'pointwise': pw_shape, 'glu': glu_shape, 'depthwise': dw_shape, 'output': tuple(z.shape)}) if trace else z

c, conv_shapes = ConvModule(D)(h, trace=True)
print(conv_shapes)
assert c.shape == h.shape

{'pointwise': (2, 64, 26), 'glu': (2, 32, 26), 'depthwise': (2, 32, 26), 'output': (2, 26, 32)}


## 4. 组合完整 encoder

设 $x_1=x+\tfrac12F_1(x)$，$x_2=x_1+A(x_1)$，$x_3=x_2+C(x_2)$，$y=\mathrm{LN}(x_3+\tfrac12F_2(x_3))$。所有残差两端均为 `[B,S,D]`。这里的每个模块是教学版；没有相对位置分数、padding mask、dropout 或训练配方。

In [7]:
class ConformerBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.ff1, self.ff2 = FeedForward(d_model), FeedForward(d_model)
        self.attn = MultiHeadSelfAttention(d_model)
        self.conv = ConvModule(d_model)
        self.final_norm = nn.LayerNorm(d_model)
    def forward(self, x):
        x = x + 0.5 * self.ff1(x)
        x = x + self.attn(x)
        x = x + self.conv(x)
        return self.final_norm(x + 0.5 * self.ff2(x))

class MiniConformer(nn.Module):
    def __init__(self, d_model=32, layers=2):
        super().__init__()
        self.sub = ConvSubsampling(d_model)
        self.pe = SinusoidalPE()
        self.blocks = nn.ModuleList([ConformerBlock(d_model) for _ in range(layers)])
    def forward(self, x, trace=False):
        x = self.pe(self.sub(x)); shapes = [tuple(x.shape)]
        for block in self.blocks:
            x = block(x); shapes.append(tuple(x.shape))
        return (x, shapes) if trace else x

model = MiniConformer(D)
y, block_shapes = model(torch.randn(B,T,FREQ,requires_grad=True), trace=True)
print(block_shapes)
assert y.shape == (B,26,D)
y.square().mean().backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
print('形状与梯度检查通过')

[(2, 26, 32), (2, 26, 32), (2, 26, 32)]
形状与梯度检查通过


**适用边界**：这是 encoder 结构和张量流练习。没有音频读取、转写头、损失函数或识别评测。继续运行第二本时会改成严格因果的卷积，以便定义可检验的流式等价条件。